# Datamine MOD2BLKS Process Example

This notebook demonstrates how to configure and run the **`mod2blks`** process wrapper in `dmstudio`.

## Process Description

## MOD2BLKS

**MOD2BLKS** creates mining block strings, reserves file and dependencies suitable for use in Studio OPs scheduler.

It is sometimes appropriate to schedule block model data rather than create more precise mining shapes for scheduling.

For example, if carrying out a pre-feasibility study, a strategic planning exercise or if the deposit is wide and tabular the model cells may be a good approximation of the shapes that would be mined and hence are suitable for scheduling.

### Input Files:

* **model** (Block Model):
  The block model from which to create the mining block outlines and reserves. This must
  be a regular block model. You should also be aware of how many cells it contains each
  model cell will produce mining block outline for use in the scheduler. If you have a
  geological resource model it is probable that you should use the
  **[REBLOCK](<reblock.md>)** process to regularise the model and increase its parent cell
  size.
  Required=Yes

### Output Files:

* **blocks** (Strings):
  The output mining block perimeter strings. These are created at the top of each model
  cell. This will contain a **BLOCKID** value that is equal to the **IJK** value of the
  model cell from which it was created. It also contains the fields **DPLUS** , **DMINUS**
  and **PFLOW** which are used by the scheduler in Studio OP.
  Required=Yes

* **results** (Table):
  The output results file. This contains the values from each cell and the **VOLUME** and
  **TONNES** of each cell.
  Required=Yes

* **depend** (Table):
  Output dependency file for use in Studio OP scheduling. This will contain the fields
  **BLOCKID1** , **BLOCKID2** , **PERCENT** and **TYPE**.
  Required=Yes

* **dependst** (Table):
  This is a string file that represents the dependencies that **MOD2BLKS** has created.
  It can be used for visualisation of the dependencies.
  Required=Yes

* **wiretr** (Wireframe triangles):
  Output mining blocks wireframe triangle file. Used for visualisation - contains
  **DEPANIM** field representing dependencies.
  Required=No

* **wirept** (Wireframe points):
  Output mining blocks wireframe point file. Used for visualisation.
  Required=No

### Fields:

* **density** (Alphanumeric : MODEL):
  The model field which contains density values. This is used to calculate mining block
  tonnages.
  Default=Undefined
  Required=No

* **phase** (Alphanumeric : MODEL):
  An optional model numeric **PHASE** field. If set this is transferred to the mining
  block reserves and strings. If not set the output **PHASE** number is 1.
  Default=Undefined
  Required=No

### Parameters:

* **density**:
  The default density to be used if the model file does not contain a density field or if
  the model density value is absent and **SETABSNT** =1.
  Range=-
  Values=-
  Default=-
  Required=No

* **setabsnt**:
  If set to 1 then absent model density values will be set to the default density value.
  The default value for **SETABSNT** is 1.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mod2blks')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mod2blks
print("Running mod2blks...")
dm_cmd.mod2blks(
    model_i='t_mod1',  # required
    blocks_o=['t_mod2blks_out'],  # required
    results_o=['t_mod2blks_out'],  # required
    depend_o='t_mod2blks_out',  # required
    dependst_o='t_mod2blks_out',  # required
    # wiretr_o='t_mod2blks_out',  # optional
    # wirept_o='t_mod2blks_out',  # optional
    # density_f='optional',  # optional
    # phase_f='optional',  # optional
    # density_p='-',  # optional
    # setabsnt_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mod2blks execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_mod2blks_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")